# EXACT Optional vLLM `/v1/models` Notebook

This notebook is separate from the real API notebook.

Use it only if you want to experiment with vLLM exposure. The main submission endpoint is `/predict`; it does not require vLLM unless organizers explicitly ask for a vLLM/OpenAI-compatible endpoint.


In [ ]:
# CELL 0 — Config

RUN_VLLM_EXPOSE = True

BASE_MODEL = "/kaggle/input/models/qwen-lm/qwen-3/transformers/8b/1"
TYPE1_LORA = "/kaggle/input/datasets/nguyenminhtric/exact-test/DATASET_DATA_TYPE_1"

print("RUN_VLLM_EXPOSE:", RUN_VLLM_EXPOSE)


In [ ]:
# CELL 1 — Install vLLM in this separate notebook only

!nvidia-smi
!pip install -q vllm

import sys, torch
print("python:", sys.executable)
print("torch:", torch.__version__)
print("cuda:", torch.cuda.is_available(), "gpu_count:", torch.cuda.device_count())

try:
    import vllm
    print("vllm:", vllm.__version__)
except Exception as e:
    print("vllm import failed:", type(e).__name__, str(e)[:1200])
    print("If this fails with XPU_KERNEL_FORMAT, stop and skip vLLM on Kaggle.")


In [ ]:
# CELL 2 — libcuda symlink fix for FlashInfer/Kaggle

!find /usr -name "libcuda.so*" 2>/dev/null | head -20
!mkdir -p /usr/local/cuda/lib64
!ln -sf /usr/local/nvidia/lib64/libcuda.so.1 /usr/local/cuda/lib64/libcuda.so
!ls -l /usr/local/cuda/lib64/libcuda.so


In [ ]:
# CELL 3 — Start vLLM and test /v1/models only

if RUN_VLLM_EXPOSE:
    import os, sys, time, subprocess, pathlib, requests, gc, torch

    subprocess.run("pkill -f vllm || true", shell=True)
    gc.collect()
    if torch.cuda.is_available():
        torch.cuda.empty_cache()
        torch.cuda.ipc_collect()
    time.sleep(5)

    print("GPU before vLLM:")
    subprocess.run("nvidia-smi", shell=True)

    log_path = "/kaggle/working/vllm_expose_only.log"
    log = open(log_path, "w")

    env = os.environ.copy()
    env["CUDA_VISIBLE_DEVICES"] = "0,1"
    env["PYTORCH_CUDA_ALLOC_CONF"] = "expandable_segments:True"
    env["LD_LIBRARY_PATH"] = "/usr/local/cuda/lib64:/usr/local/nvidia/lib64:" + env.get("LD_LIBRARY_PATH", "")

    cmd = [
        sys.executable, "-m", "vllm.entrypoints.openai.api_server",
        "--model", BASE_MODEL,
        "--dtype", "half",
        "--tensor-parallel-size", "2",
        "--enable-lora",
        "--lora-modules", f"exact_type1={TYPE1_LORA}",
        "--served-model-name", "qwen3-8b-exact",
        "--host", "0.0.0.0",
        "--port", "8000",
        "--max-model-len", "512",
        "--gpu-memory-utilization", "0.80",
        "--max-num-seqs", "1",
        "--max-num-batched-tokens", "512",
        "--enforce-eager",
        "--disable-custom-all-reduce",
        "--generation-config", "vllm",
        "--no-enable-prefix-caching",
        "--no-enable-chunked-prefill",
    ]

    print("CMD:", " ".join(cmd))
    p = subprocess.Popen(cmd, stdout=log, stderr=log, env=env)
    print("vLLM pid:", p.pid)

    ready = False
    for i in range(36):
        time.sleep(10)
        try:
            r = requests.get("http://127.0.0.1:8000/v1/models", timeout=5)
            print("READY", r.status_code)
            print(r.text[:2000])
            ready = True
            break
        except Exception as e:
            print("wait", i, repr(e))

    print("\n=== PROCESS ===")
    subprocess.run("ps aux | grep -E 'vllm|api_server' | grep -v grep || echo no_vllm_process", shell=True)

    print("\n=== NVIDIA ===")
    subprocess.run("nvidia-smi", shell=True)

    print("\n=== LOG TAIL ===")
    print(pathlib.Path(log_path).read_text()[-12000:])

    if not ready:
        print("WARNING: vLLM did not become ready. This does NOT invalidate /predict API submission.")
else:
    print("RUN_VLLM_EXPOSE=False; skipped.")


In [ ]:
# CELL 4 — Stop vLLM

import subprocess, time
subprocess.run("pkill -f vllm || true", shell=True)
time.sleep(3)
!nvidia-smi
